# Learning guide notebook: from ProbLog to probabilistic circuits

This notebook goes with the doc *Learning guide: from ProbLog to probabilistic circuits*.
Each section matches a step in the doc. Read the step there, write down your guess for each
exercise, then run the cell here to check it.

Steps 1-5 need plain Python only. Steps 6-7 need `torch` and `tididi` (the tididi env).
Run the cells from top to bottom: later steps reuse functions defined earlier.

In [ ]:
from fractions import Fraction as F
from collections.abc import Callable, Hashable, Iterator, Sequence
from itertools import product
from typing import Union

World = dict[str, bool]


def show(p: F) -> None:
    """Print a probability as an exact fraction and as a decimal."""
    print(f"{p}  (= {float(p):.4f})")

## Step 1: the two probability rules

* Independent things **multiply**.
* Things that can't happen together **add**.

In [ ]:
# Exercise 1a: two fair coins both land heads (independent: multiply)
show(F(1, 2) * F(1, 2))

# Exercise 1b: one die shows a 1 or a 2 (can't both happen: add)
show(F(1, 6) + F(1, 6))

In [ ]:
# Exercise 1c: two coins, at least one head.
# Adding is WRONG here, because both coins can be heads at once (the cases overlap):
show(F(1, 2) + F(1, 2))
# Right: 1 minus the chance of no heads at all
show(1 - F(1, 2) * F(1, 2))

## Step 2: ProbLog and possible worlds

The program:

```prolog
0.1::burglary.
0.2::earthquake.
alarm :- burglary.
alarm :- earthquake.
query(alarm).
```

`worlds` lists every true/false combination of the uncertain facts with its probability.
`prob` adds up the worlds where the query holds. This is the **brute-force checker**: slow,
but simple enough to check by reading.

In [ ]:
def worlds(facts: dict[str, F]) -> Iterator[tuple[World, F]]:
    """Every true/false assignment of the facts, with its probability."""
    names = list(facts)
    for values in product([False, True], repeat=len(names)):
        world = dict(zip(names, values))
        p = F(1)
        for name, value in world.items():
            p *= facts[name] if value else 1 - facts[name]
        yield world, p


def prob(
    facts: dict[str, F],
    query: Callable[[World], bool],
    evidence: Callable[[World], bool] = lambda w: True,
) -> F:
    """P(query | evidence), by adding up the matching worlds."""
    both = sum((p for w, p in worlds(facts) if evidence(w) and query(w)), F(0))
    given = sum((p for w, p in worlds(facts) if evidence(w)), F(0))
    return both / given

In [ ]:
def prob(
    facts: dict[str, F],
    query: Callable[[World], bool],
    evidence: Callable[[World], bool] = lambda w: True,
) -> F:
      """P(query | evidence), by adding up the matching worlds."""
      both = F(0)
      given = F(0)
      for w, p in worlds(facts):
          ev = evidence(w)
          q = query(w)
          if ev:
              given += p
              if q:
                  both += p
          print(f"world: {w}  p: {p} ({float(p):.4f})  evidence: {ev}  query: {q}  "
                f"both: {both}  given: {given}")
      print(f"P = both / given = {both} / {given} = {both / given} ({float(both / given):.4f})")
      return both / given

In [ ]:
# Exercise 2a: list the four worlds, then P(alarm)
facts = {"burglary": F("0.1"), "earthquake": F("0.2")}
alarm = lambda w: w["burglary"] or w["earthquake"]

for w, p in worlds(facts):
    print(w, p, "<- alarm" if alarm(w) else "")
show(prob(facts, alarm))

In [ ]:
# Exercise 2b: add  0.5::hears_alarm.  and  calls :- alarm, hears_alarm.
facts2 = {**facts, "hears_alarm": F("0.5")}
calls = lambda w: alarm(w) and w["hears_alarm"]
show(prob(facts2, calls))

In [ ]:
# Exercise 2c: the alarm went off. How likely is a burglary now?  P(burglary | alarm)
show(prob(facts, lambda w: w["burglary"], evidence=alarm))

In [ ]:
# Optional: the same program in real ProbLog (needs `pip install problog`).
model = """
0.1::burglary.
0.2::earthquake.
alarm :- burglary.
alarm :- earthquake.
query(alarm).
"""
try:
    from problog import get_evaluatable
    from problog.program import PrologString

    print(get_evaluatable().create_from(PrologString(model)).evaluate())
except ImportError:
    print("ProbLog isn't installed: paste the model into ProbLog's online editor instead.")

## Step 3: exactly-one choices (annotated disjunctions)

In ProbLog, `0.7::digit(0); 0.2::digit(1); 0.1::digit(2).` means *exactly one* of these is true.
To keep the worlds few, each image here can only be a 0, 1 or 2.

In [ ]:
def choice_worlds(choices: dict[str, dict[int, F]]) -> Iterator[tuple[dict[str, int], F]]:
    """choices: {group: {option: probability}}. Every way to pick one option per group."""
    groups = list(choices)
    for picks in product(*(choices[g].items() for g in groups)):
        world = {g: option for g, (option, _) in zip(groups, picks)}
        p = F(1)
        for _, q in picks:
            p *= q
        yield world, p


digits = {"image1": {0: F("0.7"), 1: F("0.2"), 2: F("0.1")},
          "image2": {0: F("0.1"), 1: F("0.8"), 2: F("0.1")}}


def p_sum(choices: dict[str, dict[int, F]], s: int) -> F:
    return sum((p for w, p in choice_worlds(choices) if w["image1"] + w["image2"] == s), F(0))


# Exercise 3a: P(sum = s) for every possible sum
for s in range(5):
    print("sum", s, ":", p_sum(digits, s), f"(= {float(p_sum(digits, s)):.4f})")

A circuit library only knows yes/no **switches**, so each choice becomes one switch per option
plus an exactly-one rule. `wmc` (weighted model counting, Step 5) adds up, over every switch
setting that satisfies a formula, the product of the switches' weights.

In [ ]:
Switch = tuple[str, int]
Assignment = dict[Switch, bool]


def wmc(
    variables: Sequence[Hashable],
    weights: dict[Hashable, tuple[F, F]],
    formula: Callable[[dict[Hashable, bool]], bool],
) -> F:
      """Weighted model count. weights[variable] = (weight when false, weight when true)."""
      total = F(0)
      for truth_values in product([False, True], repeat=len(variables)):
          assignment = dict(zip(variables, truth_values))
          if formula(assignment):
              assignment_weight = F(1)
              for variable in variables:
                  assignment_weight *= weights[variable][assignment[variable]]
              total += assignment_weight
      return total


switches = [(image, digit) for image in ("image1", "image2") for digit in range(3)]

def exactly_one(assignment: Assignment, image: str) -> bool:
    return sum(assignment[(image, digit)] for digit in range(3)) == 1


def sum_is(target_sum: int, with_rule: bool = True) -> Callable[[Assignment], bool]:
  def formula(assignment: Assignment) -> bool:
    one_digit_per_image = exactly_one(assignment, "image1") and exactly_one(assignment, "image2")
    digits_add_up = any(
        assignment[("image1", first_digit)] and assignment[("image2", target_sum - first_digit)]
        for first_digit in range(3)
        if 0 <= target_sum - first_digit < 3
    )
    return digits_add_up and (one_digit_per_image or not with_rule)
  return formula


# Exercise 3b: switch encoding, weight p when on and 1 when off. Same answer as 3a?
switch_weights = {(image, digit): (F(1), digits[image][digit]) for image, digit in switches}
show(wmc(switches, switch_weights, sum_is(1)))

In [ ]:
# Exercise 3c: what if "off" weighs 1 - p, like a normal fact?
coins = {(image, digit): (1 - digits[image][digit], digits[image][digit]) for image, digit in switches}
show(wmc(switches, coins, sum_is(1)))

# Exercise 3d: what if we forget the exactly-one rule?
show(wmc(switches, switch_weights, sum_is(1, with_rule=False)))

## Step 4: why circuits

A circuit here is nested tuples: `("lit", variable, value)`, `("and", left, right)`, `("or", left, right)`.
`evaluate` multiplies at AND and adds at OR. That is only right if the circuit is:

* **decomposable**: the two sides of every AND use different variables (so they are independent);
* **deterministic**: no setting satisfies both sides of any OR (so they never overlap).

In [ ]:
# A circuit is ("lit", variable, value) or ("and" | "or", left, right).
Circuit = Union[tuple[str, str, bool], tuple[str, "Circuit", "Circuit"]]
Weights = dict[str, tuple[F, F]]


def evaluate(c: Circuit, weights: Weights) -> F:
    if c[0] == "lit":
        return weights[c[1]][c[2]]
    left, right = evaluate(c[1], weights), evaluate(c[2], weights)
    return left * right if c[0] == "and" else left + right


def variables(c: Circuit) -> set[str]:
    return {c[1]} if c[0] == "lit" else variables(c[1]) | variables(c[2])


def satisfies(c: Circuit, a: dict[str, bool]) -> bool:
    if c[0] == "lit":
        return a[c[1]] == c[2]
    left, right = satisfies(c[1], a), satisfies(c[2], a)
    return (left and right) if c[0] == "and" else (left or right)


def decomposable(c: Circuit) -> bool:
    if c[0] == "lit":
        return True
    sides_ok = decomposable(c[1]) and decomposable(c[2])
    return sides_ok and (c[0] != "and" or not (variables(c[1]) & variables(c[2])))


def deterministic(c: Circuit) -> bool:
    """Checked by brute force over the circuit's variables."""
    if c[0] == "lit":
        return True
    if not (deterministic(c[1]) and deterministic(c[2])):
        return False
    if c[0] == "or":
        vs = sorted(variables(c))
        for values in product([False, True], repeat=len(vs)):
            a = dict(zip(vs, values))
            if satisfies(c[1], a) and satisfies(c[2], a):
                return False
    return True


def report(name: str, c: Circuit, weights: Weights) -> None:
    print(f"{name:8} value {str(evaluate(c, weights)):6}  decomposable {decomposable(c)!s:5}  "
          f"deterministic {deterministic(c)}")


w = {"burglary": (F("0.9"), F("0.1")), "earthquake": (F("0.8"), F("0.2"))}
b, e = ("lit", "burglary", True), ("lit", "earthquake", True)
not_b = ("lit", "burglary", False)

# Exercise 4a: alarm = burglary OR earthquake, as a circuit
report("naive", ("or", b, e), w)

# Exercise 4b: alarm = burglary OR (not burglary AND earthquake)
report("fixed", ("or", b, ("and", not_b, e)), w)

# Exercise 4c: burglary AND burglary
report("twice", ("and", b, b), w)

## Step 5: weighted model counting

For a decomposable, deterministic circuit, `evaluate` (fast: one pass over the circuit) equals
`wmc` (slow: every setting of every variable). Probabilities are weights `(1 - p, p)`.

In [ ]:
# Exercise 5a: the fixed alarm circuit, fast way and slow way
fixed = ("or", b, ("and", not_b, e))
vs = ["burglary", "earthquake"]
show(evaluate(fixed, w))
show(wmc(vs, w, lambda a: satisfies(fixed, a)))

In [ ]:
# Exercise 5b (harder): a variable the circuit never mentions.
# Circuit "x" over the variables x and y, with "off" weighing 1 (as in Step 3).
w_free = {"x": (F(1), F("0.3")), "y": (F(1), F("0.6"))}
just_x = ("lit", "x", True)
show(evaluate(just_x, w_free))
show(wmc(["x", "y"], w_free, lambda a: satisfies(just_x, a)))

## Step 6: DeepProbLog

Swap fixed probabilities for a network's output. Below, two "networks" are reduced to their raw
outputs (`logits`) for images that can be 0, 1 or 2; softmax turns them into probabilities.
The label says the sum is 1.

In [ ]:
import torch

logits1 = torch.tensor([2.0, 0.7, 0.0], requires_grad=True)  # image 1: leans towards 0
logits2 = torch.tensor([0.0, 2.1, 0.0], requires_grad=True)  # image 2: leans towards 1


def p_sum1() -> torch.Tensor:
    p1, p2 = logits1.softmax(0), logits2.softmax(0)
    return p1[0] * p2[1] + p1[1] * p2[0]  # the sum-1 circuit: two ways, each a product

# Exercise 6a: P(sum = 1) and the gradients of the loss
loss = -p_sum1().log()
loss.backward()
print("P(sum = 1) =", round(p_sum1().item(), 4))
print("image 1 logit gradients:", logits1.grad)
print("image 2 logit gradients:", logits2.grad)

In [ ]:
# Exercise 6b: one step of training. Does P(sum = 1) go up?
with torch.no_grad():
    logits1 -= 0.5 * logits1.grad
    logits2 -= 0.5 * logits2.grad
print("after one step: P(sum = 1) =", round(p_sum1().item(), 4))

In [ ]:
logits1 = torch.tensor([2.0, 0.7, 0.0], requires_grad=True)  # image 1: leans towards 0
logits2 = torch.tensor([0.0, 2.1, 0.0], requires_grad=True)  # image 2: leans towards 1

def p_sum4() -> torch.Tensor:
    p1, p2 = logits1.softmax(0), logits2.softmax(0)
    return p1[2] * p2[2]


# Exercise 6a: P(sum = 1) and the gradients of the loss
loss = -p_sum4().log()
loss.backward()
print("P(sum = 4) =", round(p_sum4().item(), 4))
print("image 1 logit gradients:", logits1.grad)
print("image 2 logit gradients:", logits2.grad)

In [ ]:
# Exercise 6b: one step of training. Does P(sum = 1) go up?
with torch.no_grad():
    logits1 -= 0.5 * logits1.grad
    logits2 -= 0.5 * logits2.grad
print("after one step: P(sum = 4) =", round(p_sum4().item(), 4))

## Step 7: tididi

tididi builds the circuit for you from Boolean operations: `|` is OR, `&` is AND, `~` is NOT.
Variables are numbers. `weighted_count` takes `{variable: (weight if false, weight if true)}`.

In [ ]:
import tididi

# Exercise 7a: the alarm program as a tididi circuit (1 = burglary, 2 = earthquake)
vt = tididi.Vtree.balanced(2)
alarm_circuit = tididi.literal(vt, 1) | tididi.literal(vt, 2)
show(alarm_circuit.weighted_count({1: (F("0.9"), F("0.1")), 2: (F("0.8"), F("0.2"))}))
print("satisfying settings:", alarm_circuit.model_count())

In [ ]:
# Exercise 7b: calls :- alarm, hears_alarm.   (3 = hears_alarm)
vt3 = tididi.Vtree.balanced(3)
calls_circuit = (tididi.literal(vt3, 1) | tididi.literal(vt3, 2)) & tididi.literal(vt3, 3)
show(calls_circuit.weighted_count({1: (F("0.9"), F("0.1")), 2: (F("0.8"), F("0.2")),
                                   3: (F("0.5"), F("0.5"))}))

In [ ]:
# Exercise 7c: two different-looking formulas, one function. tididi can tell.
one_way = tididi.literal(vt, 1) | tididi.literal(vt, 2)
other_way = ~(~tididi.literal(vt, 1) & ~tididi.literal(vt, 2))
print("equivalent:", one_way.equivalent(other_way))

### Checking the addition example the way you'd check anything I build

Compare the example's 19 tididi circuits with the possible-worlds answer from Step 3, for
random probabilities, exactly (fractions, no rounding). Loads `../scripts/deepproblog_addition.py`.

In [ ]:
# Exercise 7d
import importlib.util
import random

spec = importlib.util.spec_from_file_location("example", "../scripts/deepproblog_addition.py")
example = importlib.util.module_from_spec(spec)
spec.loader.exec_module(example)
queries = example.compile_queries()

rng = random.Random(1)


def random_distribution() -> list[F]:
    raw = [rng.randint(1, 9) for _ in range(10)]
    return [F(r, sum(raw)) for r in raw]


p1, p2 = random_distribution(), random_distribution()
weights = {example.var(0, d): (F(1), p1[d]) for d in range(10)}
weights |= {example.var(1, d): (F(1), p2[d]) for d in range(10)}
ten_digits = {"image1": dict(enumerate(p1)), "image2": dict(enumerate(p2))}

for s in range(19):
    assert queries[s].weighted_count(weights) == p_sum(ten_digits, s), f"sum {s} differs"
print("All 19 circuits match the possible-worlds answer exactly.")